# Model Execution: The Executor

In the previous notebook, we saw how the Scheduler acts as a master coordinator, juggling incoming requests and deciding who gets to use the GPU's precious memory and compute resources. But the Scheduler itself doesn't run the model; it just makes the plan. In this notebook, we'll explore the **Executor**, the component that takes the Scheduler's plan and makes it a reality.

By the end of this notebook, you will be able to describe the Executor's role as the "engine room" of the system, responsible for loading the LLM and performing the actual forward pass on scheduled batches. You will understand the crucial interface between the high-level scheduling logic and the low-level model inference operations, seeing how the system separates planning from doing.

In [ ]:
# --- Setup Cell ---
# This cell contains all the imports we'll need for this notebook.
# We'll build everything else from scratch.

import dataclasses
from typing import List, Dict, Any

# For our visualization
import graphviz

# A simple placeholder for a tensor. In reality, this would be a torch.Tensor or similar.
Tensor = Any

# A mapping from a block number to its corresponding KV cache data (a Tensor).
KVCache = Dict[int, Tensor]

# A mapping from a sequence's unique ID to the new token generated for it.
ExecutionOutputs = Dict[int, int]

### The Core Idea: The Kitchen Staff

Imagine a high-end, incredibly busy restaurant. The Scheduler is the *maître d'*. They manage the reservations, look at the queue of waiting guests, and expertly decide which party goes to which table to maximize the restaurant's efficiency. They create a "seating plan" for the next few minutes.

But the maître d' doesn't cook the food. That's the job of the kitchen.

The **Executor** is the restaurant's kitchen staff. The maître d' hands them a list of orders for a set of tables (a "batch"). The kitchen staff doesn't need to know about the reservation list or who is waiting outside. Their job is singularly focused: take the orders, grab the necessary ingredients (the KV cache), use the kitchen equipment (the GPU and the LLM weights), and cook the dishes (generate the next tokens).

Once the dishes are ready, they are handed back out, and the maître d' is notified. The kitchen is now ready for the next batch of orders. This separation of duties is key:

*   **Scheduler (Maître d'):** Handles complex, stateful, long-term planning. (Who gets resources? When? Which request is most important?)
*   **Executor (Kitchen Staff):** Handles stateless, short-term, intensive computation. (Given these inputs, run the model *now*.)

This design makes the system clean and modular. The Scheduler can focus on clever queuing strategies, while the Executor can focus on raw computational efficiency.

In [ ]:
# --- A Minimal Implementation of the Executor ---

# To represent a sequence's state passed from the Scheduler to the Executor.
@dataclasses.dataclass
class SequenceData:
    seq_id: int
    prompt_token_ids: List[int]
    kv_cache_blocks: List[int] # The physical block numbers assigned to this sequence.

# The batch of work the Scheduler has prepared.
@dataclasses.dataclass
class ScheduledBatch:
    sequences: List[SequenceData]

# A mock LLM that just pretends to do a forward pass.
class MockLLM:
    def forward(self, batch: ScheduledBatch, kv_cache: KVCache) -> ExecutionOutputs:
        """
        Pretends to run a forward pass.
        For each sequence, it "generates" the next token by simply incrementing the last one.
        It also "updates" the KV cache.
        """
        outputs = {}
        for seq in batch.sequences:
            last_token = seq.prompt_token_ids[-1]
            new_token = last_token + 1
            outputs[seq.seq_id] = new_token
            
            # Simulate updating the KV cache for this sequence's blocks
            for block_num in seq.kv_cache_blocks:
                kv_cache[block_num] = f"Updated KV data for seq {seq.seq_id} at token {new_token}"
        return outputs

# The Executor itself.
class Executor:
    def __init__(self):
        # In reality, this would load a massive model with billions of parameters onto the GPU.
        self.model = MockLLM()
        print("Executor initialized and model 'loaded'.")

    def execute_model(self, batch: ScheduledBatch, kv_cache: KVCache) -> ExecutionOutputs:
        """
        Takes a batch from the scheduler and executes the model.
        """
        print(f"\nExecutor: Running model on a batch of {len(batch.sequences)} sequences.")
        # The core operation: pass the batch and cache to the model.
        outputs = self.model.forward(batch, kv_cache)
        return outputs

### Walking Through the Implementation

Let's break down our simple `Executor` line by line.

1.  **`SequenceData` and `ScheduledBatch`**: These are our data structures. They represent the "contract" or interface between the Scheduler and the Executor. The Scheduler bundles up everything the Executor needs to know about the work to be done—which sequences to process, their current token history, and exactly which memory blocks (KV cache blocks) they are allowed to use—into a `ScheduledBatch` object.

2.  **`MockLLM`**: This is our stand-in for a real Large Language Model like Llama 3. Its `forward` method simulates the core computation.
    *   It receives the `ScheduledBatch` and the global `kv_cache`.
    *   It iterates through each sequence in the batch.
    *   For demonstration, its "generation logic" is trivial: `new_token = last_token + 1`. A real model would perform trillions of calculations to predict the next token.
    *   It also simulates writing new key-value pairs into the cache blocks assigned to that sequence.
    *   It returns a dictionary mapping each sequence's ID to its newly generated token.

3.  **`Executor` Class**: This is the main component.
    *   `__init__(self)`: The constructor initializes the `Executor`. In a real system, this is a heavy operation where the multi-gigabyte model weights are loaded from disk into GPU memory. We simulate this with `self.model = MockLLM()`.
    *   `execute_model(self, batch, kv_cache)`: This is the heart of the Executor. It's the entry point called by the system (specifically, the `EngineCore`) to run one step of inference. It takes the `ScheduledBatch` prepared by the Scheduler and the current state of the `kv_cache`, passes them directly to the model for computation, and returns the results.

The key takeaway is the simplicity of the Executor's main job: it's a stateless pass-through component. It receives a self-contained package of work, executes it, and returns the result, without needing to know anything about request priorities, memory fragmentation, or other complex scheduling concerns.

In [ ]:
# --- Demonstration: Running a Batch ---

# 1. Initialize the system components
executor = Executor()
# The global KV cache, managed by the Scheduler but used by the Executor.
# Let's say blocks 10, 11, 25, 30 are currently free.
global_kv_cache: KVCache = {
    1: "KV data for seq 1, token 500",
    2: "KV data for seq 1, token 501",
    15: "KV data for seq 3, token 80"
}
print(f"Initial KV Cache state: {global_kv_cache}")

# 2. The Scheduler runs and creates a batch.
# It decides to run two sequences in this step.
# - Sequence 1 is ongoing (decoding). It's assigned blocks 1 and 2.
# - Sequence 4 is brand new (prompt processing). The scheduler assigns it new blocks 10 and 11.
seq1 = SequenceData(seq_id=1, prompt_token_ids=[500, 501], kv_cache_blocks=[1, 2])
seq4 = SequenceData(seq_id=4, prompt_token_ids=[10, 20, 30], kv_cache_blocks=[10, 11])

batch_from_scheduler = ScheduledBatch(sequences=[seq1, seq4])
print("\nScheduler: Created a batch with seqs 1 and 4.")


# 3. The Executor receives the batch and runs the model.
execution_results = executor.execute_model(batch_from_scheduler, global_kv_cache)

# 4. The results are returned.
print(f"\nExecutor: Finished execution. Results: {execution_results}")
print(f"Final KV Cache state: {global_kv_cache}")

### Going Deeper: Why Separate the Executor at All?

A fair question to ask is: why have a separate `Executor` class? Why doesn't the `Scheduler` just call the model's `forward` method directly?

The answer lies in a powerful software design principle: **separation of concerns**, which leads to modularity and flexibility.

In a complex system like vLLM, the Executor might not just be a simple class. It could be an entire subsystem responsible for highly specialized tasks. The most important one is **distributed execution**. A large model might not fit on a single GPU. It needs to be split across multiple GPUs, possibly even multiple machines. This is called *tensor parallelism*.

The logic for managing communication between GPUs (e.g., using libraries like `torch.distributed`), synchronizing computations, and ensuring the model behaves as a single unit is incredibly complex. This low-level, hardware-specific logic has *nothing* to do with the high-level logic of request prioritization and memory block management.

By creating an `Executor` abstraction, vLLM allows for different backend implementations:
*   **`GPUExecutor`**: Manages running the model on one or more GPUs on a single machine, handling all the CUDA kernels and data transfers.
*   **`CPUExecutor`**: A (hypothetical) version that could run the model on the CPU for debugging or small-scale use.
*   **`DistributedExecutor`**: A version that orchestrates model execution across multiple machines in a cluster.

The Scheduler's code doesn't change. It continues to create `ScheduledBatch` objects and hand them off. The specific `Executor` implementation handles the messy details of *how* and *where* that batch is actually computed. This design keeps the Scheduler's code clean and makes it easy for developers to add support for new hardware backends without touching the core scheduling logic.

In [ ]:
# --- Visualization: The Executor's Role in the System ---

# This diagram shows the flow of data and control.
# The Scheduler makes the plan, and the Executor carries it out.

dot = graphviz.Digraph('ExecutorFlow', comment='Executor Data Flow')
dot.attr('node', shape='box', style='rounded,filled', fillcolor='lightblue')
dot.attr('edge', fontsize='10')

# Define the main components
dot.node('Scheduler', 'Scheduler\n(The Maître d\')')
dot.node('Executor', 'Executor\n(The Kitchen Staff)', fillcolor='lightgreen')
dot.node('LLM', 'LLM Model Weights\n(The Recipe & Ovens)')
dot.node('KVCache', 'KV Cache Memory\n(The Pantry)')
dot.node('EngineCore', 'EngineCore\n(The Restaurant Manager)')

# Define the flow
dot.edge('EngineCore', 'Scheduler', label=' User Requests ')
dot.edge('Scheduler', 'EngineCore', label=' Scheduled Batch ')
dot.edge('EngineCore', 'Executor', label=' execute_model(batch) ')


# Interactions during execution
dot.edge('Executor', 'LLM', label='Token IDs & Attention Mask')
dot.edge('LLM', 'Executor', label='New Logits / Tokens')
dot.edge('Executor', 'KVCache', label='Read/Write Past Keys & Values')

# Return path
dot.edge('Executor', 'EngineCore', label=' Execution Outputs ')
dot.edge('EngineCore', 'Scheduler', label=' Update Sequence State ')


# Display the graph
dot

### Summary: What We've Built

In this notebook, we've demystified the role of the Executor. We've seen that it's not a planner or a strategist, but a pure workhorse. It takes a detailed plan from the Scheduler and executes it, acting as the bridge between high-level logic and the low-level, computationally-intensive work of running a transformer model.

#### Key Takeaways:
*   **Executor's Responsibility**: To load the model weights and execute the model's forward pass on a pre-defined batch of sequences.
*   **Stateless Operation**: The Executor doesn't need to remember request history or priorities. It receives a self-contained `ScheduledBatch` for each step, performs the computation, and returns the result.
*   **Separation of Concerns**: Decoupling the Executor from the Scheduler is a critical design choice. It allows the Scheduler to focus on complex queuing and memory management, while the Executor focuses on efficient, hardware-specific model execution (like handling multi-GPU parallelism).

#### What's Next?
The Executor's job is made possible by the clever memory management system that the Scheduler uses. The `kv_cache_blocks` that we passed around are the heart of this system. In the next notebook, **"Memory Efficiency: PagedAttention KV Cache"**, we will dive deep into how this system works, revealing the technique that allows vLLM to achieve such high throughput.